# Benchmark Auswertung: Static vs. Dynamic × NIH vs. Kaggle Pneumonia

Dieses Notebook wird ausgeführt, wenn alle drei Laptops ihre Ergebnisse in results/laptop1/, results/laptop2/, results/laptop3/ gepusht haben.

Es liest alle CSVs ein und erstellt die finalen Vergleichsplots.

#### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from pathlib import Path

PROCESS_COUNTS = [1, 2, 4, 8]
IMAGE_COUNTS   = [100, 500, 1000, 5000]

Path("results").mkdir(exist_ok=True)

#### 1. Daten laden

In [ ]:
all_csvs = sorted(glob.glob("results/laptop*/results_*_laptop*.csv"))
print(f"Gefundene CSVs: {len(all_csvs)}")
for f in all_csvs:
    print(f"  {f}")

df = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)
print(f"\nZeilen gesamt: {len(df)}")
df.groupby(["Laptop", "Variante", "Datensatz"])["Bilder"].count()

#### 2. Laufzeit bei steigender Bildanzahl (alle Laptops)

In [ ]:
datasets = ["NIH", "Kaggle Pneumonia"]
laptops  = sorted(df["Laptop"].unique())

colors  = {"Static": "steelblue", "Dynamic": "darkorange"}
markers = {"Static": "o",         "Dynamic": "s"}
lines   = {1: "-", 2: "--", 3: ":"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Laufzeit bei steigender Bildanzahl (8 Prozesse, alle Laptops)")

for ax, ds in zip(axes, datasets):
    for laptop in laptops:
        for variant in ["Static", "Dynamic"]:
            sub = df[
                (df["Laptop"]   == laptop) &
                (df["Variante"] == variant) &
                (df["Datensatz"] == ds) &
                (df["Prozesse"] == 8)
            ]
            if not sub.empty:
                ax.plot(
                    sub["Bilder"], sub["Laufzeit in s"],
                    color=colors[variant],
                    marker=markers[variant],
                    linestyle=lines.get(laptop, "-"),
                    label=f"{variant} L{laptop}"
                )
    ax.set_title(ds)
    ax.set_xlabel("Bildanzahl")
    ax.set_ylabel("Laufzeit in s")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("results/runtime_by_image_count_all_laptops.png", dpi=150)
plt.show()

#### 3. Speedup (alle Laptops)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Speedup bei 2, 4 und 8 Prozessen (1000 Bilder, alle Laptops)")

for ax, ds in zip(axes, datasets):
    for laptop in laptops:
        for variant in ["Static", "Dynamic"]:
            sub = df[
                (df["Laptop"]    == laptop) &
                (df["Variante"]  == variant) &
                (df["Datensatz"] == ds) &
                (df["Bilder"]    == 1000)
            ]
            if not sub.empty:
                ax.plot(
                    sub["Prozesse"], sub["Speedup"],
                    color=colors[variant],
                    marker=markers[variant],
                    linestyle=lines.get(laptop, "-"),
                    label=f"{variant} L{laptop}"
                )
    ax.plot(PROCESS_COUNTS, PROCESS_COUNTS, "k--", alpha=0.3, label="Ideal")
    ax.set_title(ds)
    ax.set_xlabel("Prozesse")
    ax.set_ylabel("Speedup")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("results/speedup_by_processes_all_laptops.png", dpi=150)
plt.show()

#### 4. Efficiency (alle Laptops)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Efficiency bei 2, 4 und 8 Prozessen (1000 Bilder, alle Laptops)")

for ax, ds in zip(axes, datasets):
    for laptop in laptops:
        for variant in ["Static", "Dynamic"]:
            sub = df[
                (df["Laptop"]    == laptop) &
                (df["Variante"]  == variant) &
                (df["Datensatz"] == ds) &
                (df["Bilder"]    == 1000)
            ]
            if not sub.empty:
                ax.plot(
                    sub["Prozesse"], sub["Efficiency"],
                    color=colors[variant],
                    marker=markers[variant],
                    linestyle=lines.get(laptop, "-"),
                    label=f"{variant} L{laptop}"
                )
    ax.axhline(1.0, color="k", linestyle="--", alpha=0.3, label="Ideal")
    ax.set_title(ds)
    ax.set_xlabel("Prozesse")
    ax.set_ylabel("Efficiency")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("results/efficiency_by_processes_all_laptops.png", dpi=150)
plt.show()

#### 5. Static vs. Dynamic (alle Laptops)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Static vs. Dynamic — Laufzeit bei 5000 Bildern, alle Laptops")

for ax, ds in zip(axes, datasets):
    for laptop in laptops:
        for variant in ["Static", "Dynamic"]:
            sub = df[
                (df["Laptop"]    == laptop) &
                (df["Variante"]  == variant) &
                (df["Datensatz"] == ds) &
                (df["Bilder"]    == 5000)
            ]
            if not sub.empty:
                ax.plot(
                    sub["Prozesse"], sub["Laufzeit in s"],
                    color=colors[variant],
                    marker=markers[variant],
                    linestyle=lines.get(laptop, "-"),
                    label=f"{variant} L{laptop}"
                )
    ax.set_title(ds)
    ax.set_xlabel("Prozesse")
    ax.set_ylabel("Laufzeit in s")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("results/static_vs_dynamic_all_laptops.png", dpi=150)
plt.show()

#### 6. Amdahls Gesetz (Theoretisch vs. Empirisch)

In [ ]:
def amdahl(p, f=0.95):
    return 1 / ((1 - f) + f / p)

p_range = np.linspace(1, 8, 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Amdahl vs. Empirisch (1000 Bilder, alle Laptops)")

for ax, ds in zip(axes, datasets):
    ax.plot(p_range, [amdahl(p) for p in p_range], "k--", alpha=0.5, label="Amdahl (f=0.95)")
    ax.plot(PROCESS_COUNTS, PROCESS_COUNTS, "gray", linestyle=":", alpha=0.3, label="Ideal (linear)")
    for laptop in laptops:
        for variant in ["Static", "Dynamic"]:
            sub = df[
                (df["Laptop"]    == laptop) &
                (df["Variante"]  == variant) &
                (df["Datensatz"] == ds) &
                (df["Bilder"]    == 1000)
            ]
            if not sub.empty:
                ax.plot(
                    sub["Prozesse"], sub["Speedup"],
                    color=colors[variant],
                    marker=markers[variant],
                    linestyle=lines.get(laptop, "-"),
                    label=f"{variant} L{laptop}"
                )
    ax.set_title(ds)
    ax.set_xlabel("Prozesse")
    ax.set_ylabel("Speedup")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("results/amdahl_all_laptops.png", dpi=150)
plt.show()

#### 7. Ergebnistabelle

In [ ]:
table = df[[
    "Laptop", "Variante", "Datensatz", "Bilder", "Prozesse",
    "Laufzeit in s", "Speedup", "Efficiency", "Throughput"
]].sort_values(["Laptop", "Variante", "Datensatz", "Bilder", "Prozesse"])

print(table.to_markdown(index=False))